# nudge-engine: set it up and test it piece by piece

Run the cells in order, top to bottom. Each step says what you should see. If a step goes red, stop and paste the output to Claude.
Everything heavy runs on Modal, so you do not need a Colab GPU. Keep this tab open while a deploy runs.

**Before you start**
1. Push `nudge-engine` to your GitHub (a private repo is fine).
2. On Modal: add a payment method (GPUs need one) and set a monthly budget under Usage and Billing.
3. Open the key icon in Colab's left bar (Secrets), switch on *Notebook access*, and add these:

| Secret | Where it comes from |
|---|---|
| `DATABASE_URL` | Neon: connection string, edited to start with `postgresql+psycopg://` |
| `API_KEY` | make a long random string (step 2 prints how) |
| `MODELS_API_KEY` | make another long random string |
| `DEEPGRAM_API_KEY` | Deepgram console, role *Member* |
| `HF_TOKEN` | Hugging Face, a *Read* token |
| `MODAL_TOKEN_ID`, `MODAL_TOKEN_SECRET` | Modal, Settings, API Tokens |
| `GITHUB_TOKEN` | only if the repo is private |
| `TWILIO_ACCOUNT_SID`, `TWILIO_AUTH_TOKEN`, `TWILIO_FROM_NUMBER`, `ALLOWED_NUMBERS` | only for phone calls, later |


In [ ]:
#@title 1. Get the code and install it
REPO_URL = "https://github.com/aidyai/nudge-engine.git"  #@param {type:"string"}

import os
from google.colab import userdata

def secret(name, default=""):
    try:
        return userdata.get(name) or default
    except Exception:  # not set, or this notebook was not given access
        return default

token = secret("GITHUB_TOKEN")  # only needed for a private repo
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

!pip install -q uv
if not os.path.exists("/content/nudge-engine"):
    !git clone -q {clone_url} /content/nudge-engine
%cd /content/nudge-engine
os.environ["PYTHONPATH"] = "/content/nudge-engine"  # every command below can import engine, evals and scripts
!uv sync -q
print("ready")


**You should see:** `ready`. If the clone fails, check `REPO_URL`, or add `GITHUB_TOKEN` for a private repo.

In [ ]:
#@title 2. Put your keys into a .env file (read from Colab Secrets)
import pathlib

NEEDED = ["DATABASE_URL", "API_KEY", "MODELS_API_KEY", "DEEPGRAM_API_KEY", "HF_TOKEN", "MODAL_TOKEN_ID", "MODAL_TOKEN_SECRET"]
OPTIONAL = ["LLM_BASE_URL", "TTS_URL", "TWILIO_ACCOUNT_SID", "TWILIO_AUTH_TOKEN", "TWILIO_FROM_NUMBER", "ALLOWED_NUMBERS"]
KEEP_OUT_OF_ENV = {"MODAL_TOKEN_ID", "MODAL_TOKEN_SECRET"}  # used only to log in, never passed on to the app

missing = [n for n in NEEDED if not secret(n)]
if missing:
    print("To make a random string:  python -c \"import secrets; print(secrets.token_urlsafe(32))\"")
    raise SystemExit("Add these in Colab Secrets (key icon), switch on Notebook access, then run this cell again: " + ", ".join(missing))

values = {n: secret(n) for n in NEEDED + OPTIONAL if n not in KEEP_OUT_OF_ENV}
values["LLM_MODEL"] = "phonellm-mini-alpha-2"
pathlib.Path(".env").write_text("".join(f"{k}={v}\n" for k, v in values.items() if v))

def read_env():
    return dict(line.split("=", 1) for line in pathlib.Path(".env").read_text().splitlines() if "=" in line)

def set_env(key, value):
    env = read_env()
    env[key] = value
    pathlib.Path(".env").write_text("".join(f"{k}={v}\n" for k, v in env.items()))
    print(f"saved {key} to .env. Add it to Colab Secrets too, so you do not have to redeploy after a restart.")

print("keys found:", ", ".join(k for k, v in values.items() if v))


In [ ]:
#@title 3. Log in to Modal
modal_id, modal_secret = secret("MODAL_TOKEN_ID"), secret("MODAL_TOKEN_SECRET")
!uv run modal token set --token-id {modal_id} --token-secret {modal_secret}
!uv run modal profile current


**You should see:** your Modal workspace name.

In [ ]:
#@title 4. Check the keys work
!uv run --env-file .env python -m scripts.check_services


**You should see:** `[ OK ]` for Deepgram, Neon, Modal and Hugging Face, and `[SKIP]` for the LLM, the TTS and the round trip (they are not deployed yet).
If Deepgram says HTTP 403, its key was made with the wrong role: make a new one with role *Member*.

In [ ]:
#@title 5. Create the tables and the 45 fake borrowers
!uv run --env-file .env python -m engine.seed --init-schema


**You should see:** `seeded 45 borrowers across 26 archetypes`.

In [ ]:
#@title 6. Give Modal the model secrets
!uv run --env-file .env sh -c 'modal secret create huggingface-secret HF_TOKEN="$HF_TOKEN" --force'
!uv run --env-file .env sh -c 'modal secret create nudge-models MODELS_API_KEY="$MODELS_API_KEY" --force'


**You should see:** two lines saying the secrets were created.

In [ ]:
#@title 7. Start deploying the brain (it takes the longest, so start it first)
import re, subprocess

URL = re.compile(r"https://[\w.-]+\.modal\.(?:run|direct)\S*")
brain_log = open("/content/brain_deploy.log", "w")
brain = subprocess.Popen(["uv", "run", "modal", "deploy", "deploy/llm.py"], stdout=brain_log, stderr=subprocess.STDOUT, text=True)
print("The brain is building on Modal. Carry on with step 8 while it works.")


In [ ]:
#@title 8. Deploy the voice
def deploy(path):
    p = subprocess.Popen(["uv", "run", "modal", "deploy", path], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        print(line, end="")
        lines.append(line)
    p.wait()
    return p.returncode, "".join(lines)

def find_url(text, app):
    return next((u.rstrip(".,)") for u in URL.findall(text) if app in u), None)

code_, out = deploy("deploy/tts.py")
if code_:
    raise SystemExit("The voice deploy failed. Paste the error above to Claude.")
voice_url = find_url(out, "nudge-tts") or input("Paste the voice URL from the output above: ").strip()
set_env("TTS_URL", voice_url)


**You should see:** a Modal URL for the voice, and `saved TTS_URL to .env`.

In [ ]:
#@title 9. Wait for the brain to finish deploying
brain.wait()
text = open("/content/brain_deploy.log").read()
print(text[-1500:])
if brain.returncode:
    raise SystemExit("The brain deploy failed. Paste the error above to Claude.")
brain_url = find_url(text, "nudge-llm") or input("Paste the brain URL from the output above: ").strip()
set_env("LLM_BASE_URL", brain_url.rstrip("/") + "/v1")


**You should see:** a Modal URL for the brain, and `saved LLM_BASE_URL to .env`. Deploying does not start a GPU yet, so it costs nothing so far.

In [ ]:
#@title 10. Test the voice, the brain and the ears
!uv run --env-file .env python -m scripts.check_services

from IPython.display import Audio
Audio("tts_check.wav")


**You should see:** `[ OK ]` on every line, and a player. Press play: it should say a sentence about twenty-five thousand naira.
The first run starts the GPUs, so the brain line can take 5 to 15 minutes. That is normal. The round trip line speaks a sentence and has Deepgram transcribe it, which proves the voice and the ears work together.

In [ ]:
#@title 11. Run the eight scripted borrowers against the real brain
!uv run modal secret create nudge-env --from-dotenv .env --force
!uv run modal run evals/run.py

**You should see:** eight conversations, then a results table: for each borrower, `rules` (must be all of them) and `task`. A rule broken ends in an error: paste it to Claude. A missed task is normal at first, it is a number to improve.

## Measure it (this is the part job posts ask about)

Three cells: how fast the brain answers, how fast the voice answers, and how reliable the agent is over many runs. Each saves a file in `evals/results/`, and the last cell turns them into tables. Every number is measured on Modal, next to the servers. For a true cold start, wait 3 minutes after your last request before running a cell.

Get the GPU prices from modal.com/pricing and type them in (dollars per hour). The cells refuse to run with a price of 0, so the cost columns can never be fake.

In [ ]:
#@title 12. Time the brain (5 requests per call, 1 to 16 calls at once)
LABEL = "rtx6000-bf16"  #@param {type:"string"}
GPU = "RTX-PRO-6000"  #@param {type:"string"}
PRICE_PER_HOUR = 0.0  #@param {type:"number"}
TOOLS = "per_state"  #@param ["per_state", "fixed"]
assert PRICE_PER_HOUR > 0, "Type the GPU price per hour from modal.com/pricing"
!uv run --env-file .env modal run evals/bench.py::llm --label {LABEL} --gpu {GPU} --price {PRICE_PER_HOUR} --tools {TOOLS}

**You should see:** `saved evals/results/<label>-llm.json`. It takes a few minutes the first time because the GPU wakes up (that wait is recorded as the cold start). To tell the story of a change, run it again with a new `LABEL`, for example `TOOLS = fixed` as the "before" and `per_state` as the "after".

In [ ]:
#@title 13. Time the voice (whole sentences, 1 to 4 at once)
VOICE_LABEL = "l4"  #@param {type:"string"}
VOICE_GPU = "L4"  #@param {type:"string"}
VOICE_PRICE_PER_HOUR = 0.0  #@param {type:"number"}
assert VOICE_PRICE_PER_HOUR > 0, "Type the GPU price per hour from modal.com/pricing"
!uv run --env-file .env modal run evals/bench.py::tts --label {VOICE_LABEL} --gpu {VOICE_GPU} --price {VOICE_PRICE_PER_HOUR}

In [ ]:
#@title 14. Run the agent evals 5 times each
EVAL_LABEL = "baseline"  #@param {type:"string"}
RUNS = 5  #@param {type:"integer"}
!uv run --env-file .env modal run evals/run.py --runs {RUNS} --label {EVAL_LABEL}

In [ ]:
#@title 15. Make the tables
!uv run python -m evals.report --write
from IPython.display import Markdown
Markdown(open("docs/performance.md").read())

**You should see:** tables with p50, p95 and p99, requests per second, cache hit rate, cost per 1,000 requests, cold start and the eval scores. Commit `docs/performance.md` and `evals/results/` so the numbers live in the repo. Keep the failures in: a table with only wins is not believed.

In [ ]:
#@title 16. Deploy the API
code_, out = deploy("deploy/api.py")
if code_:
    raise SystemExit("The API deploy failed. Paste the error above to Claude.")
API_URL = (find_url(out, "nudge-api") or input("Paste the API URL from the output above: ").strip()).rstrip("/")

import requests
print(requests.get(API_URL + "/health", timeout=120).json())
print("\nYour website needs these two settings:\n  ENGINE_URL     =", API_URL, "\n  ENGINE_API_KEY = (your API_KEY secret)")


**You should see:** `{'status': 'ok'}`, and the two settings for your website.

In [ ]:
%%bash
# who the demo borrower is, and the birthday you will be asked for
uv run --env-file .env python - <<'EOF'
import os
from sqlalchemy import create_engine, text
row = create_engine(os.environ["DATABASE_URL"]).connect().execute(
    text("select full_name, birth_day_month from lender.borrowers where archetype = 'cooperative_due_3d' order by id limit 1")).first()
print("Answer as:", row[0], "| birthday (day-month):", row[1])
EOF


In [ ]:
#@title 17. Start a call and talk to the agent in text
import requests

H = {"X-API-Key": read_env()["API_KEY"]}
call = requests.post(API_URL + "/calls", headers=H, timeout=600).json()
print("AGENT:", call["reply"])

def say_to_agent(text):
    r = requests.post(f"{API_URL}/calls/{call['call_id']}/turns", headers=H, json={"text": text}, timeout=300)
    r.raise_for_status()
    out = r.json()
    print("YOU:  ", text)
    print("AGENT:", out["reply"])
    print("      step:", out["state"], "| verified:", out["verified"], "| outcome:", out["outcome"])

say_to_agent("Yes, speaking.")


**You should see:** the agent greet you, then reply. Keep going in the next cell with `say_to_agent("...")`: give the birthday from the cell above, then say you can pay on Friday, then say yes at the read-back.
The step, verified and outcome lines show the rules working.

In [ ]:
#@title 18. Hear the agent speak
r = requests.post(API_URL + "/speech/say", headers=H, json={"text": call["reply"]}, timeout=300)
r.raise_for_status()
open("agent.wav", "wb").write(r.content)

from IPython.display import Audio
Audio("agent.wav")


## Phone calls (only after Twilio is set up)

Do this once everything above is green. Twilio needs its own sign-up first: see the setup guide.

In [ ]:
#@title 19. Give the API your Twilio settings
needed = ["TWILIO_ACCOUNT_SID", "TWILIO_AUTH_TOKEN", "TWILIO_FROM_NUMBER", "ALLOWED_NUMBERS"]
if not all(secret(n) for n in needed):
    raise SystemExit("Add these Colab Secrets first: " + ", ".join(needed))
for n in needed:
    set_env(n, secret(n))
!uv run modal secret create nudge-env --from-dotenv .env --force
code_, out = deploy("deploy/api.py")  # redeploy so the API picks up the new secret


In [ ]:
#@title 20. Ring one of your numbers
TO = "+234"  #@param {type:"string"}
r = requests.post(API_URL + "/phone/calls", headers=H, json={"to": TO}, timeout=60)
print(r.status_code, r.json())


**You should see:** a short note saying who to answer as and what birthday to say. Your phone rings. A trial account may play a short Twilio message first and ask you to press a key.

In [ ]:
#@title Look at the logs (any time)
!uv run modal app logs nudge-api


## When all of this is green: hand over to GitHub Actions

1. In the `nudge-engine` repo on GitHub: Settings, Secrets and variables, Actions, and add `MODAL_TOKEN_ID`, `MODAL_TOKEN_SECRET`, `HF_TOKEN` and `MODELS_API_KEY`.
2. Push to `main`. Watch the Actions tab: the tests run, then the voice, the brain and the API redeploy by themselves.
3. In Vercel, set `ENGINE_URL` (the API URL from step 12) and `ENGINE_API_KEY` (your `API_KEY`).

The settings secret `nudge-env` stays something you make from this notebook (step 11). Re-run that step whenever you change a key.